# Chapter 38: Embedding Quality — Benchmarking Embeddings

A Python evaluation program that compares two embedding models using labelled sentence pairs.
It will calculate:
- Similarity scores for each pair.
- Pair ranking accuracy — whether the model scores the more-related pair higher.
- Average similarity for related and unrelated pairs.
- Encoding time for each model.

In [11]:
test_pairs = [
    {
        "text_a": "I love machine learning.",
        "text_b": "I enjoy studying artificial intelligence.",
        "label": 1
    },
    {
        "text_a": "The bank approved my loan.",
        "text_b": "My loan request was accepted.",
        "label": 1
    },
    {
        "text_a": "The cat is sleeping.",
        "text_b": "A kitten is resting.",
        "label": 1
    },
    {
        "text_a": "I love machine learning.",
        "text_b": "The football match was exciting.",
        "label": 0
    },
    {
        "text_a": "The bank approved my loan.",
        "text_b": "I enjoy cooking pasta.",
        "label": 0
    },
    {
        "text_a": "The cat is sleeping.",
        "text_b": "The stock market closed higher.",
        "label": 0
    }
]

Here, label = 1 means related, and label = 0 means unrelated.
and we will use cosine_ similarity function again to measure 


In [9]:
import time
import numpy as np

In [2]:
def cosine_similarity(vector_a, vector_b):
    dot_product = np.dot(vector_a, vector_b)

    mag_A = np.linalg.norm(vector_a)
    mag_B = np.linalg.norm(vector_b)

    if mag_A == 0 or mag_B == 0:
        raise ValueError("Cannot compare a zero vector")

    similarity = dot_product/(mag_B * mag_A)
    return float(similarity)

In [24]:
def evaluate_embedding_model(embedding_model, test_pairs):
    results = []
    start_time = time.perf_counter()

    for item in test_pairs:
        embeddings = embedding_model.encode([item["text_a"],
                                             item["text_b"]])

        score = cosine_similarity(embeddings[0], embeddings[1])
        results.append({"text_a" : item["text_a"],
                        "text_b" : item["text_b"],
                        "label" : item["label"],
                        "score" : score
                        })
    elapsed = time.perf_counter()-start_time

    return {
        "results": results,
        "encoding_time": round(elapsed, 4)
    }

In [16]:
from sentence_transformers import SentenceTransformer

model_small = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

model_large = SentenceTransformer(
    "sentence-transformers/all-mpnet-base-v2"
)

In [25]:
mini_results = evaluate_embedding_model(model_small, test_pairs)

In [26]:
mini_results

{'results': [{'text_a': 'I love machine learning.',
   'text_b': 'I enjoy studying artificial intelligence.',
   'label': 1,
   'score': 0.5693295001983643},
  {'text_a': 'The bank approved my loan.',
   'text_b': 'My loan request was accepted.',
   'label': 1,
   'score': 0.8063848614692688},
  {'text_a': 'The cat is sleeping.',
   'text_b': 'A kitten is resting.',
   'label': 1,
   'score': 0.5603188276290894},
  {'text_a': 'I love machine learning.',
   'text_b': 'The football match was exciting.',
   'label': 0,
   'score': 0.07419078797101974},
  {'text_a': 'The bank approved my loan.',
   'text_b': 'I enjoy cooking pasta.',
   'label': 0,
   'score': 0.08955197036266327},
  {'text_a': 'The cat is sleeping.',
   'text_b': 'The stock market closed higher.',
   'label': 0,
   'score': 0.046594101935625076}],
 'encoding_time': 0.7333}

In [ ]:
# Time to check which model is fast 

In [27]:
mini_results = evaluate_embedding_model(model_small, test_pairs)
mpnet_results = evaluate_embedding_model(model_large, test_pairs)

print("MiniLM time:", mini_results["encoding_time"])
print("MPNet time:", mpnet_results["encoding_time"])

MiniLM time: 0.21
MPNet time: 1.9385
